In [1]:
import numpy as np
from sklearn.externals.array_api_extra.testing import override
from model_wrapper import *
import cuml.accel
from math import floor
cuml.accel.install()

# of Training Instances: 47
# of Testing Instances: 11
Current RAM usage: 298.43 MB


In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.decomposition import IncrementalPCA
from sklearn.kernel_approximation import RBFSampler

class SGDModel(Model):
    def __init__(self, anatomical_plane, fluid_sensitive=None, fat_suppression=None, pca_n_comp=20, rbf_n_comp=25):
        self.pca_n_comp = pca_n_comp
        self.rbf_n_comp = rbf_n_comp
        self.inc_pca = None
        self.rbf_sampler = None
        self.model = OneVsRestClassifier(SGDClassifier(loss="log_loss", random_state=42))
        super().__init__(anatomical_plane, fluid_sensitive, fat_suppression, full_train=False)

    @override
    def batch_fit(self, training_folders: pd.Series, y: np.ndarray):
        if len(training_folders) <= self.pca_n_comp:
            self.inc_pca = IncrementalPCA(n_components=len(training_folders))
            n_batches = 1
        else:
            self.inc_pca = IncrementalPCA(n_components=self.pca_n_comp)
            n_batches = floor(len(training_folders) / self.pca_n_comp)

        self.rbf_sampler = RBFSampler(gamma=0.1, n_components=min(len(training_folders), self.rbf_n_comp), random_state=42)
        img_count = training_folders.apply(get_img_count)
        training_folders = training_folders[img_count >= MIN_IMG_COUNT]

        for folders_batch in np.array_split(training_folders, n_batches):
            X_batch = np.array([get_training_instance(f) for f in folders_batch])
            X_batch = np.reshape(X_batch, shape=(X_batch.shape[0], X_batch.shape[1] * X_batch.shape[2] * X_batch.shape[3]))
            self.inc_pca.partial_fit(X_batch)

        rbf_training_batch =
        for folders_batch in np.array_split(training_folders, n_batches):
            X_batch = np.array(get_training_instance(f) for f in folders_batch)
    @override
    def full_fit(self, x: np.ndarray, y: np.ndarray):
        x = np.reshape(x, shape=(x.shape[0], x.shape[1] * x.shape[2] * x.shape[3]))
        n_batches = floor(x.shape[0] / self.pca_n_comp)

        for X_batch in np.array_split(x, n_batches):
            self.inc_pca.partial_fit(X_batch)

        x_reduced = self.inc_pca.transform(x)
        self.model.fit(x_reduced, y)

    @override
    def predict_batch(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(x.shape[0], x.shape[1] * x.shape[2] * x.shape[3]))
        x_reduced = self.inc_pca.transform(x)
        return self.model.predict(x_reduced)

    @override
    def predict_instance(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(1, x.shape[0] * x.shape[1] * x.shape[2]))
        x_reduced = self.inc_pca.transform(x)
        pred_ = self.model.predict(x_reduced)
        return np.reshape(pred_, shape=(pred_.shape[1]))